# Software Architecture Model Fine-Tuning

This notebook fine-tunes `Qwen/Qwen2.5-Coder-7B-Instruct` on the software-architecture dataset using 4-bit QLoRA.

## Workflow

1. Install the Python, CUDA, Transformers, dataset, and PEFT dependencies.
2. Configure Hugging Face access. The token is read from the environment or entered securely when prompted.
3. Load the quantized base model and attach trainable LoRA adapter weights.
4. Download and split the software-architecture JSONL dataset into training and evaluation sets.
5. Format and tokenize the examples for causal language-model training.
6. Train the adapter with `Trainer`. Checkpoints and the final adapter are saved under `./results`.
7. Evaluate the trained adapter and inspect the saved files.
8. Upload `./results` to Hugging Face if the adapter should be reused elsewhere.

The files in `./results` are adapter weights, not a standalone merged model. An application can load them together with the base model using `peft`, without training again.

In [2]:
# Bootstrap pip in the venv (it was created without pip) and install uv

import sys



!{sys.executable} -m ensurepip --upgrade

%pip install --upgrade pip uv ipywidgets



# NVIDIA's CUDA dependency wheels are large, so allow slow transfers to finish.

%env UV_HTTP_TIMEOUT=600

!{sys.executable} -m uv pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132
!{sys.executable} -m uv pip install transformers datasets accelerate scikit-learn peft bitsandbytes

Looking in links: /tmp/tmpg9l1z3bj
Note: you may need to restart the kernel to use updated packages.
env: UV_HTTP_TIMEOUT=600
Using Python 3.14.4 environment at: .env
Checked 2 packages in 29ms
Using Python 3.14.4 environment at: .env
Checked 6 packages in 6ms


In [ ]:
import os
from getpass import getpass

# Disable the optional Xet transport for more reliable Hugging Face downloads.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "warning"

# Read the token from the environment or prompt without storing it in the notebook.
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HV_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token: ")
os.environ["HF_TOKEN"] = HF_TOKEN

print("Hugging Face configuration is ready.")

In [ ]:
# Confirm that the Transformers installation can download and run a small pipeline.
!python -c "from transformers import pipeline; print(pipeline('sentiment-analysis')('hugging face is the best'))"

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
config.json: 100%|████████████████████████████| 629/629 [00:00<00:00, 3.77MB/s]
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from transformers import pipeline; print(pipeline('sentiment-analysis')('hugging face is the best'))
                                             ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "/home/james/Software-Arch-AI-Model/.env/lib/python3.14/site-packages/transformers/pipelines/__init__.py", line 1027, in pipeline
    if isinstance(dtype, str

In [ ]:
import os
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"
hf_token = os.environ.get("HF_TOKEN") or None

# Load the base model in 4-bit precision so LoRA fine-tuning fits on a local GPU.
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map={"": 0},
    quantization_config=quantization_config,
    dtype=torch.float16,
    token=hf_token,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# Train only small adapter weights; the original base model remains frozen.
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
import os
from datasets import Features, Value, load_dataset

# Use the JSONL file explicitly because the repository metadata has a narrower schema.
hf_token = os.environ.get("HF_TOKEN") or None
features = Features(
    {
        "instruction": Value("string"),
        "input": Value("string"),
        "output": Value("string"),
        "nextQuestion": Value("string"),
    }
)
raw_dataset = load_dataset(
    "ajibawa-2023/Software-Architecture",
    data_files={"train": "Software_Architecture_Final.jsonl"},
    features=features,
    token=hf_token,
)

# Limit the local experiment to a reproducible subset and hold out evaluation data.
all_examples = raw_dataset["train"].shuffle(seed=42)
all_examples = all_examples.select(range(min(1000, len(all_examples))))
split = all_examples.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Training examples: {len(train_dataset)}")
print(f"Evaluation examples: {len(eval_dataset)}")

In [ ]:
from transformers import DataCollatorForLanguageModeling

def format_examples(batch):
    texts = []
    for instruction, input_text, output, next_question in zip(
        batch["instruction"],
        batch["input"],
        batch["output"],
        batch["nextQuestion"],
    ):
        texts.append(
            f"Instruction:\n{instruction}\n\n"
            f"Input:\n{input_text}\n\n"
            f"Response:\n{output}\n\n"
            f"Next question:\n{next_question}"
        )
    return {"text": texts}

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# Convert structured dataset records into causal language-model training examples.
formatted_train = train_dataset.map(
    format_examples,
    batched=True,
    remove_columns=train_dataset.column_names,
)
formatted_eval = eval_dataset.map(
    format_examples,
    batched=True,
    remove_columns=eval_dataset.column_names,
)
tokenized_train = formatted_train.map(tokenize, batched=True, remove_columns=["text"])
tokenized_eval = formatted_eval.map(tokenize, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("Dataset formatting and tokenization complete.")

In [ ]:
import torch
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    processing_class=tokenizer,
)

# Train the adapter and save checkpoints so a crash does not lose the run.
trainer.train()
trainer.save_model("./results")
tokenizer.save_pretrained("./results")
print("Training complete and adapter saved to ./results.")

In [ ]:
from pathlib import Path

# Evaluate the saved adapter and list the files available for reuse or upload.
eval_results = trainer.evaluate()
print(f"Evaluation loss: {eval_results['eval_loss']:.4f}")
print("Saved adapter files:", sorted(path.name for path in Path("./results").glob("*")))
from huggingface_hub import HfApi

# Upload the adapter; the base model remains referenced by adapter_config.json.
HfApi().upload_folder(
    folder_path="./results",
    repo_id="JamesAHowieson/SoftwareArchitecture",
)

In [4]:
import torch

# Run a small inference check against the trained adapter.
model.eval()
model.config.use_cache = True

messages = [
    {
        "role": "system",
        "content": "You are a senior software architect. Give practical, concise recommendations.",
    },
    {
        "role": "user",
        "content": (
            "Design a maintainable architecture for a multi-tenant document collaboration "
            "platform with authentication, real-time editing, search, and audit logging."
        ),
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

NameError: name 'model' is not defined